<a href="https://colab.research.google.com/github/Laylacheng/NLP_2026_/blob/main/NN_%E4%B8%AD%E6%96%87%E6%96%87%E6%9C%AC%E5%88%86%E9%A1%9E.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NN-自然語言處理
## 教學目標
- 本教學著重於自然語言處理，主要涵蓋 `RNN`，並且以 [`LSTM`](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) 為範例。
- 這份教學的目標是介紹如何以 Python 和 PyTorch 實作神經網路。

## 使用 NN 來進行中文的分類任務

- 我們將在這個教學裡讓大家實作中文情緒分析（Sentiment Analysis）
- 本資料集爲外賣平臺用戶評價分析，[下載連結](https://raw.githubusercontent.com/SophonPlus/ChineseNlpCorpus/master/datasets/waimai_10k/waimai_10k.csv)。
- 資料集欄位爲標籤（label）和評價（review），
- 標籤 1 爲正向，0 爲負向。
- 正向 4000 條，負向約 8000 條。

In [ ]:
# 0. 下載資料與安裝 jieba

!mkdir -p data
!wget https://raw.githubusercontent.com/SophonPlus/ChineseNlpCorpus/master/datasets/waimai_10k/waimai_10k.csv -O data/waimai_10k.csv
!pip install jieba

--2026-03-08 14:14:48--  https://raw.githubusercontent.com/SophonPlus/ChineseNlpCorpus/master/datasets/waimai_10k/waimai_10k.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 919380 (898K) [text/plain]
Saving to: ‘data/waimai_10k.csv’

data/waimai_10k.csv 100%[===================>] 897.83K  --.-KB/s    in 0.008s  

2026-03-08 14:14:48 (114 MB/s) - ‘data/waimai_10k.csv’ saved [919380/919380]



In [ ]:
# 1. 導入所需套件

# 第3方套件
import jieba
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch
from torch.utils.data import DataLoader
from torch.nn.utils.rnn import (
    pad_sequence,
    pack_padded_sequence,
    pad_packed_sequence,
)
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

# typing hints
from typing import Dict, List, Tuple
from pandas import DataFrame
from torch import Tensor

/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:44: SyntaxWarning: invalid escape sequence '\.'
  re_han_default = re.compile("([\u4E00-\u9FD5a-zA-Z0-9+#&\._%\-]+)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/__init__.py:46: SyntaxWarning: invalid escape sequence '\s'
  re_skip_default = re.compile("(\r\n|\s)", re.U)
/usr/local/lib/python3.12/dist-packages/jieba/finalseg/__init__.py:78: SyntaxWarning: invalid escape sequence '\.'
  re_skip = re.compile("([a-zA-Z0-9]+(?:\.\d+)?%?)")


In [ ]:
# 2. 以 pandas 讀取資料
# 請先下載資料集

df = pd.read_csv("./data/waimai_10k.csv")

In [ ]:
# 3. 觀察資料

df.head()

,label,review
0,1,很快，好吃，味道足，量大
1,1,没有送水没有送水没有送水
2,1,非常快，态度好。
3,1,方便，快捷，味道可口，快递给力
4,1,菜味道很棒！送餐很及时！


## 建立字典
- 電腦無法僅透過字符來區分不同字之間的意涵
- 電腦視覺領域依賴的是影像資料本身的像素值
- 我們讓電腦理解文字的方法是透過向量
- 文字的意義藉由向量來進行表達的形式稱為 word embeddings
- 舉例:
$\textrm{apple}=[0.123, 0.456,0.789,\dots,0.111]$

- 如何建立每個文字所屬的向量？
- 在那之前，要先建立分散式字詞的字典
    - 可粗分兩種斷詞方式 (tokenization):
        1. 每個字都斷 (character-level)
        2. 斷成字詞 (word-level)

## Word embeddings
- 著名的方法有:
    1. word2vec: Skip-gram, CBOW (continuous bag-of-words)
    2. GloVe
    3. fastText
- 本教學使用 PyTorch 內建的 Embedding 層來實作 word embeddings

In [ ]:
word_to_idx = {
    "<pad>": 0,
    "<unk>": 1,
    "好吃": 2,
    "棒": 3,
    "给力": 4,
}
embeds = torch.nn.Embedding(5, 5) # 5個字、每個字維度為5

In [ ]:
def get_word_id(word: str, vocab: Dict[str, int], unk_idx: int = 1):
    return vocab.get(word, unk_idx)

In [ ]:
lookup_tensor = torch.tensor(
    [
        get_word_id("好吃", word_to_idx),
        get_word_id("不棒", word_to_idx),
        get_word_id("<unk>", word_to_idx),
    ],
)
word_embed = embeds(lookup_tensor)
print(word_embed)

tensor([[ 0.4195, -0.6881,  0.9578,  0.6472,  0.4393],
        [-0.4618,  0.0042, -0.5537, -0.4931, -0.4328],
        [-0.4618,  0.0042, -0.5537, -0.4931, -0.4328]],
       grad_fn=<EmbeddingBackward0>)


### 複習 torch.nn.Linear 用法

In [ ]:
# torch.nn.Linear 用法範例
# m 是一個線性轉換層，之前我們都在 forward 裡面使用它

m = torch.nn.Linear(20, 30)
input = torch.randn(128, 20) # 假設有 128 筆資料，每筆資料有 20 維
output = m(input)

print(output.size())

torch.Size([128, 30])


In [ ]:
# 4. 建立字典
use_jieba=True

vocab = {'<pad>':0, '<unk>':1}

if use_jieba:
    words = []
    for sent in df['review']:
        tokens = jieba.lcut(sent, cut_all=False)
        words.extend(tokens)

else:
    # 以 character-level 斷詞
    words = df['review'].str.cat()

# 使字詞不重複
words = sorted(set(words))
for idx, word in enumerate(words):
    # 一開始已經放兩個進去 dictionary 了
    idx = idx + 2
    # 將 word to id 放到 dictionary
    vocab[word] = idx

# 查看字典大小
print("The vocab size is {}.".format(len(vocab)))

Building prefix dict from the default dictionary ...
DEBUG:jieba:Building prefix dict from the default dictionary ...
Dumping model to file cache /tmp/jieba.cache
DEBUG:jieba:Dumping model to file cache /tmp/jieba.cache
Loading model cost 3.809 seconds.
DEBUG:jieba:Loading model cost 3.809 seconds.
Prefix dict has been built successfully.
DEBUG:jieba:Prefix dict has been built successfully.


The vocab size is 11010.


## 使用 PyTorch 建立 Dataset
![Imgur](https://i.imgur.com/wGnfCmH.png)

In [ ]:
# 5. 將資料分成 train/ validation/ test

train_data, test_data = train_test_split(
    df,
    test_size=0.2,
)
train_data, validation_data = train_test_split(
    train_data,
    test_size=0.1,
)

In [ ]:
# 6. 定義超參數

parameters = {
    "padding_idx": 0,
    "vocab_size": len(vocab),
    # Hyperparameters
    "embed_dim": 300,
    "hidden_dim": 256,
    "module_name": 'rnn',
    "num_layers": 2,
    "learning_rate": 5e-4,
    "epochs": 10,
    "max_seq_len": 50,
    "batch_size": 64,
    "bidirectional": True,
}

In [ ]:
# 7. 建立 PyTorch Dataset (定義 class)

class WaimaiDataset(torch.utils.data.Dataset):
    # 繼承 torch.utils.data.Dataset
    def __init__(
        self,
        vocab: Dict[str, int],
        data: DataFrame,
        max_seq_len: int,
        use_jieba: bool,
    ) -> None:
        self.df = data
        self.max_seq_len = max_seq_len
        # 可以選擇要不要使用結巴進行斷詞
        self.use_jieba = use_jieba
        self.vocab = vocab
        self.unk_idx = self.vocab.get('<unk>')

    # 改寫繼承的 __getitem__ function
    def __getitem__(self, idx: int) -> Tuple[Tensor, Tensor]:
        # dataframe 的第一個 column 是 label
        # dataframe 的第一個 column 是 評論的句子
        label, sent = self.df.iloc[idx, 0:2]
        # 先將 label 轉為 float32 以方便後面進行 loss function 的計算
        label_tensor = torch.tensor(label, dtype=torch.float32)
        if self.use_jieba:
            # 使用 lcut 可以 return list
            tokens = jieba.lcut(sent, cut_all=False)
        else:
            # 每個字都斷詞
            tokens = list(sent)

        # 控制最大的序列長度
        tokens = tokens[:self.max_seq_len]

        # 根據 vocab 轉換 word id
        # 如果找不到該字詞，就用 <unk> 的 index 來表示
        tokens_id = [
            self.vocab.get(word, self.unk_idx) for word in tokens
        ]
        tokens_tensor = torch.LongTensor(tokens_id)

        # 所以 第 0 個index是句子，第 1 個index是 label
        return tokens_tensor, label_tensor

    # 改寫繼承的 __len__ function
    def __len__(self) -> int:
        return len(self.df)

In [ ]:
# 8. 建立 PyTorch Dataset (執行 class)
use_jieba=use_jieba

trainset = WaimaiDataset(
    vocab,
    train_data,
    parameters["max_seq_len"],
    use_jieba=use_jieba
)
validset = WaimaiDataset(
    vocab,
    validation_data,
    parameters["max_seq_len"],
    use_jieba=use_jieba
)
testset = WaimaiDataset(
    vocab,
    test_data,
    parameters["max_seq_len"],
    use_jieba=use_jieba
)

In [ ]:
# 9. 整理 batch 的資料 (定義 function)

def collate_batch(batch: List[Tuple[Tensor, Tensor]]) -> Tuple[Tensor, Tensor, Tensor]:
    # 抽每一個 batch 的第 0 個(注意順序)
    text = [i[0] for i in batch]

    # 紀錄每個句子的原始長度 (padding 前)
    lengths = torch.LongTensor([len(seq) for seq in text])

    # 進行 padding
    text = pad_sequence(text, batch_first=True)

    # 抽每一個 batch 的第 1 個(注意順序)
    label = [i[1] for i in batch]
    # 把每一個 batch 的答案疊成一個 tensor
    label = torch.stack(label)

    return text, label, lengths

In [ ]:
# 10. 建立資料分批 (mini-batches)

# 因為會針對 trainloader 進行 shuffle
# 對 trainloader 進行 shuffle 有助於降低 overfitting

trainloader = DataLoader(
    trainset,
    batch_size=parameters["batch_size"],
    collate_fn=collate_batch,
    shuffle=True,
)
validloader = DataLoader(
    validset,
    batch_size=parameters["batch_size"],
    collate_fn=collate_batch,
    shuffle=False,
)
testloader = DataLoader(
    testset,
    batch_size=parameters["batch_size"],
    collate_fn=collate_batch,
    shuffle=False,
)

## 建立模型
![Imgur](https://i.imgur.com/OgLBBm7.png)
- 模型建置的流程如上圖所示
- 文字的部份會透過 Dataset 及 DataLoader 進行處理
- embedding 層經由 nn.embedding 來實現 embedding lookup 的功能
- embedding 層再接上模型，最後接上分類層，即可進行分類任務
- 本範例提供的 Model class 可以藉由更換 module_name 來呼叫不同的 RNN

### LSTM 的輸出
1. `output`: 假設使用 `batch_first=True`，則 `output` 的形狀為 `(B, S, D * Hout​)`
    - `B`: batch_size
    - `S`: sequence_length
    - `D`: 是否 bidirectional (是：2；否：1)
    - `Hout`: hidden_size
    - 表示 **每個時間步** 的 hidden state 輸出
2. `h_n`: 每一層在**最後一個有效時間步**的 hidden state，形狀為 `(D * num_layers, B, Hout)`
    - 如果是單向 LSTM，`h_n` 可以理解成「每一層最後一個字處理完後的 hidden state」
    - 如果是雙向 LSTM，則每一層都會有 forward 和 backward 各一個最後 hidden state
    - 常用來做分類任務，因為它可以代表整個句子的最終表示

3. `c_n`: 每一層在**最後一個有效時間步**的 cell state，形狀為 `(D * num_layers, B, Hout)`
    - `c_n` 是 LSTM 內部的記憶狀態（cell state）
    - `h_n` 是對外輸出的 hidden state
    - 可以把 `c_n` 想成「LSTM 內部保留下來的長期記憶」
    - 可以把 `h_n` 想成「目前這一步拿來輸出的狀態」

In [ ]:
# 11. 建立 RNN 模型 (定義 class)

class RNNModel(torch.nn.Module):
    def __init__(
        self,
        vocab_size: int,
        embed_dim: int,
        hidden_dim: int,
        num_layers: int,
        padding_idx: int,
        bi: bool = False,
    ) -> None:
        super().__init__()
        self.hidden_dim = hidden_dim
        self.bidirectional = bi

        # 定義 Embedding 層
        self.embedding = torch.nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embed_dim,
            padding_idx=padding_idx
        )
        # 定義 LSTM
        self.rnn = torch.nn.LSTM(
            input_size=embed_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=bi
        )
        # 根據是否 bidirectional 決定輸出層的輸入維度
        direction_factor = 2 if bi else 1
        self.fc = torch.nn.Linear(
            # 如果 bi-directional，hidden_dim 是兩倍
            in_features=hidden_dim * direction_factor,
            out_features=1
        )

    def forward(self, X: Tensor, lengths: Tensor) -> Tuple[Tensor, Tensor]:
        """定義神經網路的前向傳遞的進行流程
        Arguments:
            - X: 輸入值，維度為(B, S)，其中 B 為 batch size，S 為 sentence length
            - lengths: 每個 batch 中的每筆句子 padding 前的真實長度，維度為(B,)
        Returns:
            - logits: 模型的輸出值，維度為(B, 1)，其中 B 為 batch size
            - Y: 模型的輸出值但經過非線性轉換 (這邊是用 sigmoid)，維度為(B, 1)，其中 B 為 batch size
        """
        # 維度: (B, S) -> (B, S, E)
        # B: batch size; S: sentence length; E: embedding dimension
        E = self.embedding(X)

        # 將 padded sequence 打包，讓 LSTM 跳過 pad 的位置
        packed_E = pack_padded_sequence(
            E,
            lengths.cpu(),          # lengths 建議放 CPU
            batch_first=True,
            enforce_sorted=False    # 這樣 collate_batch 不用先排序
        )

        # LSTM 處理 packed sequence
        packed_out, (h_n, c_n) = self.rnn(packed_E)

        # 如果你還想保留每個時間步的輸出，可再 unpack
        # H_out, _ = pad_packed_sequence(packed_out, batch_first=True)

        if self.bidirectional:
            # h_n shape: (num_layers * num_directions, B, hidden_dim)
            # 取最後一層 forward 與 backward hidden state
            forward_last = h_n[-2]
            backward_last = h_n[-1]
            combined = torch.cat([forward_last, backward_last], dim=1)
        else:
            # 取最後一層 hidden state
            combined = h_n[-1]

        logits = self.fc(combined)
        Y = torch.sigmoid(logits)

        return logits, Y

### 關於 Padding
#### 1. 我們需要 Padding
在一個 batch 中，不同句子的長度通常不一樣。  
例如：

- 第 1 句：`abc`
- 第 2 句：`x`

如果要把它們一起組成 tensor，形狀必須整齊一致；  
但這兩句長度分別是 3 和 1，無法直接堆疊成同一個 batch。

因此我們需要 **padding**，把較短的句子補到和最長句子一樣長，例如：

- 第 1 句：`abc`
- 第 2 句：`x<pad><pad>`

這樣才能把它們整理成同一個 tensor，一起送進模型做運算。

#### 2. 我們需要 [`pack_padded_sequence`](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html#torch.nn.utils.rnn.PackedSequence)
如果直接把上述的資料送進 LSTM，模型其實也會去處理 `<pad>`，  
但 `<pad>` 不是真正的文字，因此會引入一些不必要的計算。

這時就可以使用 `pack_padded_sequence`，所以最後 PyTorch 接收到的資訊會變成

- `data = axbc`
- `batch_sizes = [2, 1, 1]`

可以理解成：

- 第一個時間步有 2 筆有效資料
- 第二個時間步有 1 筆有效資料
- 第三個時間步有 1 筆有效資料

In [ ]:
# 12. 執行訓練所需要的準備工作

model = RNNModel(
    vocab_size=parameters["vocab_size"],
    embed_dim=parameters["embed_dim"],
    hidden_dim=parameters["hidden_dim"],
    num_layers=parameters["num_layers"],
    padding_idx=parameters["padding_idx"],
    bi=parameters["bidirectional"],
)
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=parameters["learning_rate"])
loss_func = torch.nn.BCEWithLogitsLoss() # 含有 sigmoid 的版本

## 設定訓練流程

In [ ]:
# 13. 設定訓練流程 (定義 function)

def train(
    trainloader: DataLoader,
    model: torch.nn.Module,
    optimizer: torch.optim.Optimizer,
    loss_func: torch.nn.Module,
) -> float:
    """定義訓練時的進行流程
    Arguments:
        - trainloader: 具備 mini-batches 的 dataset，由 PyTorch DataLoader 所建立
        - model: 要進行訓練的模型
        - optimizer: 最佳化目標函數的演算法
    Returns:
        - train_loss: 模型在一個 epoch 的 training loss
    """
    # 設定模型的訓練模式
    model.train()

    # 記錄一個 epoch中 training 過程的 loss
    train_loss = 0
    # 從 trainloader 一次一次抽
    for x, y, lengths in tqdm(trainloader, desc="Training"):
        # 將變數丟到指定的裝置位置
        x = x.to(device)
        y = y.to(device)

        # 重新設定模型的梯度
        optimizer.zero_grad()

        # 1. 前向傳遞 (Forward Pass)
        logits, pred = model(x, lengths)

        # 2. 計算 loss (loss function 為二元交叉熵)
        loss = loss_func(logits.squeeze(-1), y)

        # 3. 計算反向傳播的梯度
        loss.backward()
        # 4. "更新"模型的權重
        optimizer.step()

        # 一個 epoch 會抽很多次 batch，所以每個 batch 計算完都要加起來
        # .item() 在 PyTorch 中可以獲得該 tensor 的數值
        train_loss += loss.item()

    return train_loss / len(trainloader)

## 設定驗證流程

In [ ]:
# 14. 設定驗證流程 (定義 function)

def evaluate(
    dataloader: DataLoader,
    model: torch.nn.Module,
    loss_func: torch.nn.Module,
) -> Tuple[float, float]:
    """定義驗證時的進行流程
    Arguments:
        - dataloader: 具備 mini-batches 的 dataset，由 PyTorch DataLoader 所建立
        - model: 要進行驗證的模型
    Returns:
        - loss: 模型在驗證/測試集的 loss
        - acc: 模型在驗證/測試集的正確率
    """
    # 設定模型的驗證模式
    # 此時 dropout 會自動關閉
    model.eval()
    total_loss = 0 # 紀錄 loss 數值
    label_list = []
    prediction_list = []

    # 設定現在不計算梯度
    with torch.no_grad():
        # 從 dataloader 一次一次抽
        for x, y, lengths in tqdm(dataloader, desc="Evaluating"):
            x, y = x.to(device), y.to(device)
            logits, pred = model(x, lengths)

            # 計算 loss (loss function 為二元交叉熵)
            # 模型輸出的維度是 (B, 1)，使用.squeeze(-1)可以讓維度變 (B,)
            loss = loss_func(logits.squeeze(-1), y)
            total_loss += loss.item()

            # 預測的數值大於 0.5 則視為類別1，反之為類別0
            pred = (pred > 0.5) * 1 # pred.shape: (B, 1)
            prediction_list.extend(pred.cpu().squeeze(-1).tolist())
            label_list.extend(y.cpu().tolist())

    avg_loss = total_loss / len(dataloader)
    # 計算正確率
    acc = accuracy_score(label_list, prediction_list)
    print()

    return avg_loss, acc

## 開始訓練

In [ ]:
# 15. 整個訓練及驗證過程的 script

train_loss_history = []
valid_loss_history = []

for epoch in range(parameters["epochs"]):
    train_loss = train(
        trainloader,
        model,
        optimizer=optimizer,
        loss_func=loss_func,
    )

    print("Training loss at epoch {} is {}.".format(epoch+1, train_loss))
    train_loss_history.append(train_loss)

    print("=====Start validation=====")
    valid_loss, valid_acc = evaluate(
        dataloader=validloader,
        model=model,
        loss_func=loss_func,
    )
    valid_loss_history.append(valid_loss)
    print(f"Validation accuracy at epoch {epoch+1} is {valid_acc}.")
    print(f"Validation loss is {valid_loss}.")

    torch.save(model.state_dict(), "model_epoch_{}.pkl".format(epoch))

Training: 100%|██████████| 135/135 [00:04<00:00, 32.46it/s]


Training loss at epoch 1 is 0.4064880481472722.
=====Start validation=====


Evaluating: 100%|██████████| 15/15 [00:00<00:00, 43.37it/s]


Validation accuracy at epoch 1 is 0.8779979144942649.
Validation loss is 0.3171636839707693.


Training: 100%|██████████| 135/135 [00:04<00:00, 28.52it/s]


...

Validation accuracy at epoch 9 is 0.8790406673618353.
Validation loss is 0.4937613785266876.


Training: 100%|██████████| 135/135 [00:04<00:00, 28.65it/s]


Training loss at epoch 10 is 0.012770255171279941.
=====Start validation=====


Evaluating: 100%|██████████| 15/15 [00:00<00:00, 41.46it/s]

Validation accuracy at epoch 10 is 0.8967674661105318.
Validation loss is 0.5705402637521426.


In [ ]:
# 16. 預測測試集

best_epoch = np.argmin(valid_loss_history)
model.load_state_dict(
    torch.load(f"model_epoch_{best_epoch}.pkl")
)

print("=====Start testing=====\n")
test_loss, test_acc = evaluate(testloader, model, loss_func)
print("Testing accuracy is {}.".format(test_acc))

=====Start testing=====



Evaluating: 100%|██████████| 38/38 [00:00<00:00, 42.56it/s]


Testing accuracy is 0.8773978315262719.
